# <center> Homework 134

In [4]:
import tensorflow as tf

## Task 0
да се тестват примерите в книгата

да се имплементират и сравнят следните стратегии:

    greedy decoding
    top k sampling
    nucleus sampling


In [ ]:
shakespeare_url = "https://homl.info/shakespeare" # shortcut URL
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)

with open(filepath) as f:
    shakespeare_text = f.read()

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [10]:
import tf_preprocessing_layers 
from importlib import reload

In [17]:
reload(tf_preprocessing_layers)
from tf_preprocessing_layers import TextVectorization 

text = 'Before, hear me speak. hear'

text_vec = TextVectorization()
text_vec.adapt(text)
text_vec.call(text)

<tf.Tensor: shape=(5,), dtype=int32, numpy=array([3, 2, 4, 5, 2], dtype=int32)>

In [18]:
text_vec = TextVectorization(split='character')
text_vec.adapt(text)
text_vec.call(text)

<tf.Tensor: shape=(25,), dtype=int32, numpy=
array([ 7,  2,  8,  9,  4,  2,  3,  6,  2,  5,  4,  3, 10,  2,  3, 11, 12,
        2,  5, 13,  3,  6,  2,  5,  4], dtype=int32)>

In [19]:
len(text)

27

In [ ]:
len(shakespeare_text)

1115394

In [ ]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character", standardize="lower")
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]

In [ ]:
encoded -= 2 # drop tokens 0 (pad) and 1 (unknown), which we will not use
n_tokens = text_vec_layer.vocabulary_size() - 2 # number of distinct chars = 39
dataset_size = len(encoded) # total number of chars = 1,115,394

In [30]:
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window_ds: window_ds.batch(length + 1))

    if shuffle:
        ds = ds.shuffle(buffer_size=100_000, seed=seed)

    ds = ds.batch(batch_size)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

In [ ]:
length = 100
tf.random.set_seed(42)

train_set = to_dataset(encoded[:1_000_000], length=length, shuffle=True, seed=42)
valid_set = to_dataset(encoded[1_000_000:1_060_000], length=length)
test_set = to_dataset(encoded[1_060_000:], length=length)

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="nadam",
              metrics=["accuracy"])

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/ml_models/my_shakespeare_model.keras",
    monitor="val_accuracy",
    save_best_only=True)

history = model.fit(train_set,
                    validation_data=valid_set,
                    epochs=10,
                    callbacks=[model_ckpt])

Epoch 1/10
  31245/Unknown 433s 13ms/step - accuracy: 0.5477 - loss: 1.4967

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


31247/31247 ━━━━━━━━━━━━━━━━━━━━ 451s 14ms/step - accuracy: 0.5477 - loss: 1.4966 - val_accuracy: 0.5306 - val_loss: 1.6090
Epoch 2/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 433s 13ms/step - accuracy: 0.5983 - loss: 1.2891 - val_accuracy: 0.5414 - val_loss: 1.5809
Epoch 3/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 489s 15ms/step - accuracy: 0.6025 - loss: 1.2705 - val_accuracy: 0.5438 - val_loss: 1.5715
Epoch 4/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 444s 14ms/step - accuracy: 0.6049 - loss: 1.2599 - val_accuracy: 0.5443 - val_loss: 1.5686
Epoch 5/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 450s 14ms/step - accuracy: 0.6059 - loss: 1.2537 - val_accuracy: 0.5433 - val_loss: 1.5665
Epoch 6/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 457s 14ms/step - accuracy: 0.6076 - loss: 1.2484 - val_accuracy: 0.5448 - val_loss: 1.5645
Epoch 7/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 477s 15ms/step - accuracy: 0.6076 - loss: 1.2463 - val_accuracy: 0.5448 - val_loss: 1.5611
Epoch 8/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 499s 15ms/step - accur

In [ ]:
shakespeare_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda X: X - 2),
    model
])

In [ ]:
y_proba = shakespeare_model.predict(tf.constant(["To be or not to b"]))[0, -1]
y_pred = tf.argmax(y_proba) # choose the most probable character ID
text_vec_layer.get_vocabulary()[y_pred + 2]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step


np.str_('e')

In [52]:
def extend_text(text, model, gen_func, n_chars=50, temperature=1, threshold=0.7, k=3):
  for _ in range(n_chars):
    text += gen_func(text, model, temperature=temperature, threshold=threshold, k=k)

  return text

def print_tensor(text):
  text = text.numpy()[0].decode("utf-8")
  text = text.replace("\\n", "\n")
  print(text)

### Temperature sampling

In [50]:
def next_char_temp(text, model, temperature=1, threshold=None, k=None):
  y_proba = model.predict([text])[0, -1:]
  rescaled_logits = tf.math.log(y_proba) / temperature
  char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
  return text_vec_layer.get_vocabulary()[char_id + 2]

In [ ]:
tf.random.set_seed(42)
text = tf.constant(["To be or not to b"])
generated = extend_text(text, next_char_temp, temperature=0.01)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━

In [ ]:
generated2 = extend_text(text, next_char_temp, temperature=1)
print_tensor(generated2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━

### greedy decoding

In [ ]:
def next_char_greedy(text, model, temperature=None, threshold=None, k=None):
  y_proba = model.predict([text])[0, -1]
  char_id = tf.argmax(y_proba)
  return text_vec_layer.get_vocabulary()[char_id + 2]

In [ ]:
generated3 = extend_text(text, next_char_greedy)
print_tensor(generated3)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━

### top k sampling

In [33]:
def next_char_top_k(text, model, k=3, temperature=1, threshold=None):
  y_proba = model.predict([text])[0, -1:]

  sort_inxs = tf.argsort(y_proba, direction='DESCENDING')
  sort_vals = tf.gather(y_proba, sort_inxs, axis=1)[0]
  top_k_proba = sort_vals[:, :k]

  rescaled_logits = tf.math.log(top_k_proba) / temperature
  char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
  char_id = sort_inxs[:, char_id][0]
  return text_vec_layer.get_vocabulary()[char_id + 2]

In [ ]:
generated = extend_text(text, next_char_top_k, k=3)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━

In [ ]:
generated = extend_text(text, next_char_top_k, k=6, temperature=0.1)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━

### nucleus sampling

In [ ]:
def next_char_nucleus(text, model, threshold=0.7, temperature=1, k=None):
  y_proba = model.predict([text])[0, -1:]

  sort_inxs = tf.argsort(y_proba, direction='DESCENDING')
  sort_vals = tf.gather(y_proba, sort_inxs, axis=1)[0]
  cumsum = tf.math.cumsum(sort_vals, axis=1)

  cutoff = tf.argmax(cumsum >= threshold, axis=-1)[0]
  top_proba_inxs = sort_inxs[:, :cutoff + 1]
  top_proba = tf.gather(y_proba, top_proba_inxs, axis=1)[0]

  rescaled_logits = tf.math.log(top_proba) / temperature
  char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
  char_id = sort_inxs[:, char_id][0]
  return text_vec_layer.get_vocabulary()[char_id + 2]

In [ ]:
generated = extend_text(text, next_char_nucleus, threshold=0.90)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━

In [ ]:
generated = extend_text(text, next_char_nucleus, threshold=0.90, temperature=10)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━

## Task 1
да се прочете блога на Andrej Karpath

    да се имплементират някои от примерите (по избор)


### Модел обучен на Python код

In [ ]:
with open('tf_model.py') as f:
    py_code = f.read()

In [ ]:
with open('tf_data.py') as f:
    py_code += f.read()

with open('dataframe.py') as f:
    py_code += f.read()

In [ ]:
print(py_code[:1000])

from collections import defaultdict
from io import StringIO
import numpy as np
from abc import ABC, abstractmethod
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.base import check_array, check_is_fitted
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, root_mean_squared_error, precision_score, mean_squared_error
from tqdm import tqdm
from scipy.special import erf
import joblib
import pickle
from copy import deepcopy
import tensorflow as tf
from functools import partial
from tf_data import Dataset



one = tf.constant(1.0)
selu_alpha = tf.constant(1.67)
selu_const = tf.constant(1.05)

# -------- activations --------

@tf.function
def sigmoid(z): 
    return 1 / (1 + tf.exp(-tf.clip_by_value(z, -500, 500)))

# @tf.function
# def softmax(z):
#     classes_exp = tf.exp(z)
#     return classes_exp / tf.reduce_sum(classes_exp, axis=1, keepdims=True)

@tf.function(jit_compile=True)
def softmax(z):
    z_max = tf

In [ ]:
len(py_code)

249371

In [ ]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character", standardize="lower")
text_vec_layer.adapt([py_code])
encoded_py_code = text_vec_layer([py_code])[0]

encoded_py_code -= 2
n_tokens = text_vec_layer.vocabulary_size() - 2
dataset_size = len(encoded_py_code)

In [ ]:
n_tokens

69

In [ ]:
length = 100
tf.random.set_seed(42)

train_set = to_dataset(encoded_py_code[:240_000], length=length, shuffle=True, seed=42)
valid_set = to_dataset(encoded_py_code[240_000:245_000], length=length)
test_set = to_dataset(encoded_py_code[245_000:], length=length)

In [ ]:
py_code_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=8),
    tf.keras.layers.GRU(64, return_sequences=True, dropout=0.5, kernel_regularizer='l2'),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])

py_code_model.compile(loss="sparse_categorical_crossentropy",
              optimizer="nadam",
              metrics=["accuracy"])

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/ml_models/py_code_model.keras",
    monitor="val_accuracy",
    save_best_only=True)

history = py_code_model.fit(train_set,
                    validation_data=valid_set,
                    epochs=10,
                    callbacks=[model_ckpt])

Epoch 1/10
   7497/Unknown 111s 12ms/step - accuracy: 0.4721 - loss: 2.1090

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


7497/7497 ━━━━━━━━━━━━━━━━━━━━ 114s 12ms/step - accuracy: 0.4721 - loss: 2.1090 - val_accuracy: 0.5744 - val_loss: 1.7548
Epoch 2/10
7497/7497 ━━━━━━━━━━━━━━━━━━━━ 110s 12ms/step - accuracy: 0.6115 - loss: 1.4868 - val_accuracy: 0.5859 - val_loss: 1.7195
Epoch 3/10
7497/7497 ━━━━━━━━━━━━━━━━━━━━ 111s 12ms/step - accuracy: 0.6305 - loss: 1.4022 - val_accuracy: 0.5946 - val_loss: 1.7060
Epoch 4/10
7497/7497 ━━━━━━━━━━━━━━━━━━━━ 142s 12ms/step - accuracy: 0.6390 - loss: 1.3630 - val_accuracy: 0.5989 - val_loss: 1.6917
Epoch 5/10
7497/7497 ━━━━━━━━━━━━━━━━━━━━ 151s 18ms/step - accuracy: 0.6449 - loss: 1.3373 - val_accuracy: 0.6021 - val_loss: 1.6774
Epoch 6/10
7497/7497 ━━━━━━━━━━━━━━━━━━━━ 111s 12ms/step - accuracy: 0.6488 - loss: 1.3208 - val_accuracy: 0.6043 - val_loss: 1.6752
Epoch 7/10
7497/7497 ━━━━━━━━━━━━━━━━━━━━ 112s 12ms/step - accuracy: 0.6519 - loss: 1.3075 - val_accuracy: 0.6028 - val_loss: 1.6650
Epoch 8/10
7497/7497 ━━━━━━━━━━━━━━━━━━━━ 111s 12ms/step - accuracy: 0.6533 - lo

In [ ]:
py_code_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda X: X - 2),
    py_code_model
])

In [ ]:
text = tf.constant(["def fit"])
generated = extend_text(text, py_code_model, next_char_temp, n_chars=300, temperature=0.5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━

In [ ]:
print_tensor(generated)

def fitl(rewfice(target_self):
                    nfwal=col_name = self.columns]
                                if not isinstance(list, layer):
                                resurn_self.columns = self._series:
                                                                                             


In [ ]:
text = tf.constant(["class Conv1D(Layer):"])
generated = extend_text(text, py_code_model, next_char_top_k, n_chars=300, temperature=0.5, k=5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━

In [ ]:
print_tensor(generated)

class Conv1D(Layer):
        if not isinstance(key2, dataframe)
                                self._series = tenpon(len(self.index, done):
                                reser_name
                    inxs = [self.columns.initializer
                                        if not isinstance(key), self.index):
      


### Модел обучен на имена

In [43]:
with open('datasets/names.txt') as f:
    names = f.read()

In [26]:
with open('datasets/names.txt') as f:
    row_names = f.readlines()

In [ ]:
row_names[:2]

['Abagael\n', 'Abagail\n']

In [28]:
row_names = [name[:-1] for name in row_names]

In [29]:
row_names[:2]

['Abagael', 'Abagail']

In [30]:
len(row_names)

7944

In [31]:
row_names = set(row_names)

In [32]:
len(row_names)

7579

In [35]:
print(names[:52])

Abagael
Abagail
Abbe
Abbey
Abbi
Abbie
Abby
Abigael
A


In [44]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character", standardize="lower")
text_vec_layer.adapt([names])
encoded_names = text_vec_layer([names])[0]

encoded_names -= 2
n_tokens = text_vec_layer.vocabulary_size() - 2
dataset_size = len(encoded_names)

In [ ]:
n_tokens

30

In [ ]:
dataset_size

55869

In [ ]:
length = 30
tf.random.set_seed(42)

train_set = to_dataset(encoded_names[:48_000], length=length, shuffle=True, seed=42)
valid_set = to_dataset(encoded_names[48_000:52_000], length=length)
test_set = to_dataset(encoded_names[52_000:], length=length)

In [ ]:
names_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=32),
    tf.keras.layers.LSTM(64, return_sequences=True, dropout=0.5),
    tf.keras.layers.LSTM(128, return_sequences=True, dropout=0.5),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])

names_model.compile(loss="sparse_categorical_crossentropy",
              optimizer="nadam",
              metrics=["accuracy"])

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/ml_models/names_model.keras",
    monitor="val_accuracy",
    save_best_only=True)

history = names_model.fit(train_set,
                    validation_data=valid_set,
                    epochs=10,
                    callbacks=[model_ckpt])

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - accuracy: 0.2378 - loss: 2.5442 - val_accuracy: 0.2855 - val_loss: 2.3991
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.4161 - loss: 1.8983 - val_accuracy: 0.3507 - val_loss: 2.2111
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - accuracy: 0.4881 - loss: 1.6643 - val_accuracy: 0.3914 - val_loss: 2.0877
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.5329 - loss: 1.5162 - val_accuracy: 0.4238 - val_loss: 2.0078
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - accuracy: 0.5608 - loss: 1.4232 - val_accuracy: 0.4248 - val_loss: 1.9936
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.5817 - loss: 1.3529 - val_accuracy: 0.4381 - val_loss: 1.9770
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - accuracy: 0.5963 - loss: 1.2988 - val_accuracy: 0.4398 - val_loss: 1.9862
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.6093 -

In [45]:
names_model = tf.keras.models.load_model('language_models/names_model.keras')

In [46]:
names_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda X: X - 2),
    names_model
])

In [48]:
def generate_name(seed, next_char_func, temperature=1, k=3):
  while True:
    next = next_char_func(seed, names_model, temperature=temperature, k=k)
    if next == '\n':
      break

    seed += next

  return seed

In [45]:
text = tf.constant(["Ma"])
generated = generate_name(text, next_char_top_k, temperature=0.5, k=5)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Marina


In [35]:
'Marina' in row_names

True

In [46]:
text = tf.constant(["G"])
generated = generate_name(text, next_char_temp, temperature=1)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Gmarbel


In [38]:
'Gmarbel' in row_names

False

In [49]:
text = tf.constant(["Z"])
generated = generate_name(text, next_char_temp, temperature=2)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Zopytus


In [36]:
'Zopytus' in row_names

False

In [56]:
text = tf.constant(["Iz"])
generated = generate_name(text, next_char_temp, temperature=1)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
Izsyn


In [57]:
'Izsyn' in row_names

False

In [61]:
text = tf.constant(["Z"])
generated = generate_name(text, next_char_top_k, k=10, temperature=2)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Zolyene


In [64]:
text = tf.constant(["E"])
generated = generate_name(text, next_char_top_k, k=10, temperature=10)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Etose


In [34]:
'Etose' in row_names

False

In [65]:
text = tf.constant(["Ro"])
generated = generate_name(text, next_char_top_k, k=20, temperature=10)
print_tensor(generated)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Romu


In [33]:
'Romu' in row_names

False